# Merge Weather Data with PM2.5 Data\nMerge PM2.5 values from yearly files (2021-2025) into each weather station file by matching date and station code.

In [2]:
import pandas as pd
import os

RAW_DIR = "rawdata"
OUT_DIR = "processed_data"

# Weather station files (area code files)
station_files = ["05T", "42T", "57T", "58T", "62T", "79T", "82T", "83T", "86T", "87T"]

# Load and combine all yearly PM2.5 files (2021-2025)
pm_frames = []
for year in range(2021, 2026):
    df = pd.read_csv(f"{RAW_DIR}/{year}.csv")
    # Keep only rows where Date looks like a valid date (e.g. 1/1/2021)
    df = df[df["Date"].astype(str).str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")]
    pm_frames.append(df)

pm_all = pd.concat(pm_frames, ignore_index=True)
pm_all["Date"] = pd.to_datetime(pm_all["Date"], format="%m/%d/%Y")
print(f"PM2.5 data: {len(pm_all)} rows, stations: {len(pm_all.columns)-1}")
pm_all.head()

PM2.5 data: 1826 rows, stations: 107


,Date,02T,05T,10T,11T,12T,59T,61T,03T,50T,...,105T,116T,117T,113T,114T,115T,118T,119T,120T,121T
0,2021-01-01,27.0,20.0,22.0,25.0,22.0,20.0,25.0,24.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-02,32.0,25.0,26.0,27.0,27.0,23.0,26.0,29.0,31.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-03,46.0,37.0,33.0,41.0,40.0,38.0,29.0,44.0,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-04,39.0,31.0,32.0,36.0,38.0,36.0,28.0,46.0,41.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-05,50.0,31.0,31.0,32.0,44.0,28.0,24.0,67.0,41.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Merge PM2.5 into each weather station file and export
for station in station_files:
    # Read weather data
    weather = pd.read_csv(f"{RAW_DIR}/{station}.csv")
    weather["date"] = pd.to_datetime(weather["date"], format="%m/%d/%Y")
    
    # Extract PM2.5 column for this station
    if station in pm_all.columns:
        pm_station = pm_all[["Date", station]].copy()
        pm_station = pm_station.rename(columns={"Date": "date", station: "pm25"})
        # Convert pm25 to numeric (N/A -> NaN)
        pm_station["pm25"] = pd.to_numeric(pm_station["pm25"], errors="coerce")
        
        # Merge on date
        merged = weather.merge(pm_station, on="date", how="left")
        
        # Export
        merged.to_csv(f"{OUT_DIR}/{station}.csv", index=False)
        matched = merged["pm25"].notna().sum()
        print(f"{station}: {len(merged)} rows, PM2.5 matched: {matched}/{len(merged)}")
    else:
        print(f"WARNING: {station} not found in PM2.5 data columns!")

print("\nDone! Files exported to", OUT_DIR)

05T: 1827 rows, PM2.5 matched: 1799/1827
42T: 1827 rows, PM2.5 matched: 1786/1827
57T: 1827 rows, PM2.5 matched: 1771/1827
58T: 1827 rows, PM2.5 matched: 1816/1827
62T: 1827 rows, PM2.5 matched: 1611/1827
79T: 1827 rows, PM2.5 matched: 1809/1827
82T: 1827 rows, PM2.5 matched: 1816/1827
83T: 1827 rows, PM2.5 matched: 1810/1827
86T: 1827 rows, PM2.5 matched: 1791/1827
87T: 1827 rows, PM2.5 matched: 1796/1827

Done! Files exported to processed_data


In [4]:
# Preview one merged file
sample = pd.read_csv(f"{OUT_DIR}/{station_files[0]}.csv")
print(f"Preview: {station_files[0]}.csv")
print(f"Columns: {list(sample.columns)}")
sample.head(10)

Preview: 05T.csv
Columns: ['date', 'dew_point_2m_mean', 'temperature_2m_mean', 'precipitation_sum', 'wind_direction_10m_dominant', 'wind_speed_10m_mean', 'surface_pressure_mean', 'relative_humidity_2m_mean', 'pm25']


,date,dew_point_2m_mean,temperature_2m_mean,precipitation_sum,wind_direction_10m_dominant,wind_speed_10m_mean,surface_pressure_mean,relative_humidity_2m_mean,pm25
0,2021-01-01,12.8,22.1,0.0,49,9.5,1014.3,56,20.0
1,2021-01-02,14.9,22.1,0.0,10,6.5,1013.5,64,25.0
2,2021-01-03,16.2,23.5,0.0,17,7.8,1012.1,65,37.0
3,2021-01-04,17.8,25.0,0.0,24,7.6,1011.8,65,31.0
4,2021-01-05,19.3,25.5,0.0,21,5.9,1011.8,69,31.0
5,2021-01-06,20.1,26.8,0.0,55,7.5,1010.1,69,32.0
6,2021-01-07,19.4,26.7,0.0,58,7.6,1009.3,66,26.0
7,2021-01-08,18.6,26.7,0.0,73,14.6,1009.6,62,21.0
8,2021-01-09,14.7,23.7,0.0,70,12.6,1011.6,58,19.0
9,2021-01-10,14.8,23.6,0.0,62,9.2,1011.6,59,27.0
